In [1]:
import os


from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

/home/poo/miniconda3/envs/scraper_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Must use the "exact same" Embedding model as when creating the database
embedding_model = "qwen3-embedding:8b"
embedding = OllamaEmbeddings(model=embedding_model)

/tmp/ipykernel_73048/1426085832.py:3: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embedding = OllamaEmbeddings(model=embedding_model)


In [3]:
chroma_db_dir = "cuda_quantum_chroma_db"

In [4]:
#Load the existing ChromaDB
vectorstore = Chroma(
    persist_directory=chroma_db_dir,
    embedding_function=embedding
)

/tmp/ipykernel_73048/1853635707.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(
E0000 00:00:1773315044.952172   73048 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1773315044.952562   73048 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1773315044.952566   73048 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1773315044.952569   73048 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pre

In [5]:
retriever = vectorstore.as_retriever(
    #mmr  algorithm for diversity context
    search_type="mmr",
    search_kwargs={"k": 8, "fetch_k": 20}
)

In [6]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [7]:
while True:
    user_input = input("Please enter your question (enter 'q' to quit): ").strip()
    if user_input.lower() in ['q', 'quit', 'exit']:
        print("Ending conversation.")
        break
    if not user_input:
        continue
        
    print("Retrieving and reranking with the local BGE model...")
    
    # Execute retrieval and reranking
    retrieved_docs = retriever.invoke(user_input)
    
    print("\n[Retrieved Relevant Materials (Top 4)]")
    for i, doc in enumerate(retrieved_docs, 1):
        source = doc.metadata.get("source", "Unknown Source")
        chunk_file = doc.metadata.get("chunk_file", "Unknown Chunk File")
        # Extract the score assigned by the reranker model
        score = doc.metadata.get("relevance_score", "N/A")
        
        print(f"--- Reference {i} (Rerank Score: {score} | Source: {source} | Chunk File: {chunk_file}) ---")
        print(doc.page_content)
        print("\n")
            
    print("="*50 + "\n")

Retrieving and reranking with the local BGE model...

[Retrieved Relevant Materials (Top 4)]
--- Reference 1 (Rerank Score: N/A | Source: api_languages_python_api.txt | Chunk File: api_languages_python_api_chunk_562.txt) ---
cudaq.initialize_cudaq()


cudaq.initialize_cudaq(\*\*kwargs) → None

Initialize the CUDA Quantum environment.



cudaq.num_available_gpus()


cudaq.num_available_gpus() → int

The number of available GPUs detected on the system.



cudaq.set_random_seed()


cudaq.set_random_seed(arg0: int) → None

Provide the seed for backend quantum kernel simulation.



Data Types


class cudaq.Target
The cudaq.Target represents the underlying infrastructure that CUDA Quantum kernels will execute on. Instances of cudaq.Target describe what simulator they may leverage, the quantum_platform required for execution, and a description for the target.


property description
A string describing the features for this cudaq.Target.



is_emulated()


is_emulated(self: cudaq.mlir._mlir_li